# Basic SIR Model History Matching

This notebook demonstrates how to use the history matching library with a stochastic SIR epidemiological model. We'll show both automated and interactive workflows using the new object-oriented API.

## Overview

History matching is a Bayesian method for model calibration that:
1. Uses statistical emulators to approximate expensive simulations
2. Iteratively reduces the parameter space to "plausible" regions
3. Avoids regions where the model cannot match observed data

We'll calibrate a SIR model to synthetic outbreak data to recover the "true" transmission parameters.

In [ ]:
# Automatically reload any modules that are changed
%load_ext autoreload
%autoreload 2

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import the history matching library with clean OOP API
import history_matching as hm

%matplotlib inline
np.random.seed(42)  # For reproducible results

## SIR Model Definition

We define a stochastic SIR model using the tau-leap algorithm. This will be our "expensive" simulation that we want to emulate.

In [ ]:
class SIR:
    """A stochastic SIR model using tau-leap algorithm."""
    
    def __init__(self, beta=1.2, gamma=0.3, s0=990, i0=10, r0=0, 
                 n_days=20, step=0.1, seed=None):
        """
        Initialize the SIR model.
        
        Args:
            beta: Transmission rate (>0)
            gamma: Recovery rate (>0)  
            s0: Initial susceptible population
            i0: Initial infected population
            r0: Initial recovered population
            n_days: Number of days to simulate
            step: Time step size in days
            seed: Random seed for reproducibility
        """
        # Validate inputs
        assert beta > 0, "Transmission rate must be positive"
        assert gamma > 0, "Recovery rate must be positive"
        assert s0 >= 0 and i0 >= 0 and r0 >= 0, "Initial conditions must be non-negative"
        assert n_days > 0, "Simulation days must be positive"
        assert 0 < step <= 1, "Step size must be between 0 and 1"
        
        self.beta = beta
        self.gamma = gamma
        self.s0 = int(s0)
        self.i0 = int(i0)
        self.r0 = int(r0)
        self.n_days = int(n_days)
        self.step = step
        self.seed = seed
        self.simulation_complete = False
    
    def run(self):
        """Run the stochastic SIR simulation."""
        # Setup
        n_steps = math.ceil((self.n_days - 1) / self.step)
        steps_per_day = int(1 / self.step)
        N = self.s0 + self.i0 + self.r0
        
        # Initialize arrays
        s = np.zeros(n_steps + 1, dtype=int)
        i = np.zeros(n_steps + 1, dtype=int)
        r = np.zeros(n_steps + 1, dtype=int)
        
        s[0] = self.s0
        i[0] = self.i0
        r[0] = self.r0
        
        # Daily outputs
        s_daily = np.zeros(self.n_days, dtype=int)
        i_daily = np.zeros(self.n_days, dtype=int)
        r_daily = np.zeros(self.n_days, dtype=int)
        
        s_daily[0] = s[0]
        i_daily[0] = i[0]
        r_daily[0] = r[0]
        
        if self.seed is not None:
            np.random.seed(self.seed)
        
        # Tau-leap simulation
        for j in range(1, n_steps + 1):
            # Calculate event rates
            lambda_transmission = (self.beta * s[j-1] * i[j-1] / N) * self.step
            lambda_recovery = self.gamma * i[j-1] * self.step
            
            # Sample events
            delta_Mt = np.random.poisson(lambda_transmission)
            delta_Mr = np.random.poisson(lambda_recovery)
            
            # Update states
            s[j] = max(s[j-1] - delta_Mt, 0)
            i[j] = max(i[j-1] + delta_Mt - delta_Mr, 0)
            r[j] = r[j-1] + delta_Mr
        
        # Extract daily values
        for j in range(1, self.n_days):
            k = j * steps_per_day
            s_daily[j] = s[k]
            i_daily[j] = i[k]
            r_daily[j] = r[k]
        
        self.s = s_daily
        self.i = i_daily
        self.r = r_daily
        self.simulation_complete = True
        
        return s_daily, i_daily, r_daily
    
    def get_incidence(self):
        """Calculate daily incidence (new infections)."""
        if not self.simulation_complete:
            self.run()
        
        incidence = np.zeros(len(self.i))
        incidence[0] = 0
        for j in range(1, len(self.i)):
            incidence[j] = self.s[j-1] - self.s[j]
        
        return incidence
    
    def plot(self, title=None):
        """Plot S, I, R trajectories."""
        if not self.simulation_complete:
            self.run()
        
        x = range(self.n_days)
        if title is None:
            title = f'SIR Model: β={self.beta:.2f}, γ={self.gamma:.2f}'
        
        plt.figure(figsize=(10, 6))
        plt.plot(x, self.s, 'g-', label='Susceptible', linewidth=2)
        plt.plot(x, self.i, 'r-', label='Infected', linewidth=2)
        plt.plot(x, self.r, 'b-', label='Recovered', linewidth=2)
        plt.legend()
        plt.xlabel('Days')
        plt.ylabel('Population')
        plt.title(title)
        plt.grid(True, alpha=0.3)
        plt.show()

## Generate Synthetic "Observed" Data

We'll create synthetic outbreak data with known parameters that we'll try to recover through history matching.

In [ ]:
# "True" parameters that generated our synthetic data
beta_true = 1.3
gamma_true = 0.5
population_size = 10000
initial_infected = 100

print(f"True parameters we want to recover:")
print(f"  β (transmission rate): {beta_true}")
print(f"  γ (recovery rate): {gamma_true}")
print(f"  R₀ (basic reproduction number): {beta_true/gamma_true:.2f}")

# Generate "observed" data
true_model = SIR(
    beta=beta_true,
    gamma=gamma_true,
    s0=population_size - initial_infected,
    i0=initial_infected,
    seed=42  # Fixed seed for reproducible "observations"
)

S_obs, I_obs, R_obs = true_model.run()
incidence_obs = true_model.get_incidence()

# Plot the "true" outbreak
true_model.plot("True Outbreak (Synthetic Observed Data)")

# Show incidence data
plt.figure(figsize=(10, 6))
plt.plot(range(len(incidence_obs)), incidence_obs, 'ko-', markersize=4, linewidth=2)
plt.xlabel('Days')
plt.ylabel('Daily Incidence')
plt.title('Observed Daily Incidence Data')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nPeak incidence: {max(incidence_obs):.0f} cases/day")
print(f"Total cases: {sum(incidence_obs):.0f}")
print(f"Attack rate: {sum(incidence_obs)/population_size:.1%}")

## Define Simulation Function for History Matching

History matching requires a function that takes parameter samples and returns simulation results.

In [ ]:
def sir_simulation_function(samples: pd.DataFrame) -> pd.DataFrame:
    """
    Simulation function for history matching.
    
    Args:
        samples: DataFrame with columns ['beta', 'gamma'] containing parameter samples
        
    Returns:
        DataFrame with simulation outputs (incidence time series)
    """
    results = []
    
    for _, row in samples.iterrows():
        # Run SIR model with these parameters
        model = SIR(
            beta=row['beta'],
            gamma=row['gamma'],
            s0=population_size - initial_infected,
            i0=initial_infected,
            # Note: no fixed seed - we want stochastic variation
        )
        
        incidence = model.get_incidence()
        
        # Create result dictionary with named features
        result = {f'incidence_day_{i}': incidence[i] for i in range(len(incidence))}
        
        # Add summary statistics as additional features
        result['peak_incidence'] = max(incidence)
        result['total_cases'] = sum(incidence)
        result['attack_rate'] = sum(incidence) / population_size
        
        results.append(result)
    
    return pd.DataFrame(results)

# Test the simulation function
test_samples = pd.DataFrame({
    'beta': [1.0, 1.5, 2.0],
    'gamma': [0.3, 0.5, 0.7]
})

test_results = sir_simulation_function(test_samples)
print(f"Test simulation completed:")
print(f"  Input shape: {test_samples.shape}")
print(f"  Output shape: {test_results.shape}")
print(f"  Output features: {list(test_results.columns[:5])}...")

## Setup History Matching Workflow

Now we configure the history matching using the new object-oriented API.

In [ ]:
# Define parameter bounds (search space)
parameter_bounds = {
    'beta': (0.5, 3.0),    # Transmission rate
    'gamma': (0.1, 1.0)    # Recovery rate
}

# Define observations - we'll focus on a few key features
# In practice, you wouldn't use all time points as this leads to high-dimensional emulation
observations = {
    'peak_incidence': (max(incidence_obs), 50),       # (mean, variance)
    'total_cases': (sum(incidence_obs), 200),
    'incidence_day_5': (incidence_obs[5], 30),        # Early epidemic
    'incidence_day_10': (incidence_obs[10], 40),      # Peak period
    'incidence_day_15': (incidence_obs[15], 20),      # Late epidemic
}

print(f"Parameter space:")
for param, (min_val, max_val) in parameter_bounds.items():
    print(f"  {param}: [{min_val}, {max_val}]")

print(f"\nObservations:")
for feature, (mean, std) in observations.items():
    print(f"  {feature}: {mean:.1f} ± {std:.1f}")

# Build history matching engine using quick setup
engine = hm.quick_setup(
    parameter_bounds=parameter_bounds,
    observations=observations,
    n_samples=500,           # Samples per iteration
    max_iterations=4,        # Maximum number of iterations
    sampling_strategy='lhs', # Latin Hypercube Sampling
    emulator_type='gpr',     # Gaussian Process Regression
    implausibility_threshold=3.0,  # Standard threshold
    random_seed=123
)

print(f"\nHistory matching engine created:")
print(f"  Parameters: {len(engine.parameter_space.get_parameter_names())}")
print(f"  Observations: {len(observations)}")
print(f"  Samples per iteration: {engine._n_samples}")
print(f"  Max iterations: {engine._max_iterations}")

In [ ]:
# Set the simulation function
engine.set_simulation_function(sir_simulation_function)
print("✅ Simulation function configured")

## Automated History Matching

Let's run the full history matching workflow automatically.

In [ ]:
print("🚀 Running automated history matching...")
print("This will run multiple iterations, training emulators and reducing the parameter space.\n")

# Run automated workflow
results = engine.run()

print(f"\n📊 History matching completed!")
print(f"  Iterations run: {len(results)}")
print(f"  Final acceptance rate: {engine.acceptance_rate:.3f}")
print(f"  Total samples generated: {engine.progress.total_samples_generated}")
print(f"  Total samples accepted: {engine.progress.total_samples_accepted}")
print(f"  Emulators trained: {engine.progress.total_emulators_trained}")

# Show iteration summary
print(f"\n📋 Iteration Summary:")
for i, result in enumerate(results, 1):
    print(f"  Iteration {i}: {len(result.samples)} samples, features {result.selected_features}")
    print(f"    Parameter ranges - β: [{result.samples['beta'].min():.2f}, {result.samples['beta'].max():.2f}], "
          f"γ: [{result.samples['gamma'].min():.2f}, {result.samples['gamma'].max():.2f}]")

## Analyze Results

Let's examine how well we recovered the true parameters.

In [ ]:
# Get final plausible parameter samples
final_result = results[-1]
final_samples = final_result.samples

print(f"📈 Final Parameter Estimates:")
print(f"  Total plausible samples: {len(final_samples)}")
print(f"  β range: [{final_samples['beta'].min():.3f}, {final_samples['beta'].max():.3f}] (true: {beta_true})")
print(f"  γ range: [{final_samples['gamma'].min():.3f}, {final_samples['gamma'].max():.3f}] (true: {gamma_true})")
print(f"  β median: {final_samples['beta'].median():.3f} (true: {beta_true})")
print(f"  γ median: {final_samples['gamma'].median():.3f} (true: {gamma_true})")

# Check if true parameters are in plausible region
beta_recovered = (final_samples['beta'].min() <= beta_true <= final_samples['beta'].max())
gamma_recovered = (final_samples['gamma'].min() <= gamma_true <= final_samples['gamma'].max())

print(f"\n✅ Parameter Recovery:")
print(f"  β correctly identified: {'Yes' if beta_recovered else 'No'}")
print(f"  γ correctly identified: {'Yes' if gamma_recovered else 'No'}")

if beta_recovered and gamma_recovered:
    print(f"  🎉 Success! Both parameters recovered in plausible region.")
else:
    print(f"  ⚠️  Some parameters not fully recovered. Consider more iterations or different features.")

In [ ]:
# Visualize parameter space reduction
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Parameter scatter plot
ax = axes[0, 0]
ax.scatter(final_samples['beta'], final_samples['gamma'], alpha=0.6, s=20, color='green')
ax.axvline(beta_true, color='red', linestyle='--', linewidth=2, label=f'True β = {beta_true}')
ax.axhline(gamma_true, color='red', linestyle='--', linewidth=2, label=f'True γ = {gamma_true}')
ax.set_xlabel('β (transmission rate)')
ax.set_ylabel('γ (recovery rate)')
ax.set_title('Final Plausible Parameter Space')
ax.legend()
ax.grid(True, alpha=0.3)

# Beta histogram
ax = axes[0, 1]
ax.hist(final_samples['beta'], bins=20, alpha=0.7, color='blue', density=True)
ax.axvline(beta_true, color='red', linestyle='--', linewidth=2, label=f'True β = {beta_true}')
ax.axvline(final_samples['beta'].median(), color='green', linestyle='-', linewidth=2, 
           label=f'Estimated β = {final_samples["beta"].median():.2f}')
ax.set_xlabel('β (transmission rate)')
ax.set_ylabel('Density')
ax.set_title('β Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Gamma histogram  
ax = axes[1, 0]
ax.hist(final_samples['gamma'], bins=20, alpha=0.7, color='orange', density=True)
ax.axvline(gamma_true, color='red', linestyle='--', linewidth=2, label=f'True γ = {gamma_true}')
ax.axvline(final_samples['gamma'].median(), color='green', linestyle='-', linewidth=2,
           label=f'Estimated γ = {final_samples["gamma"].median():.2f}')
ax.set_xlabel('γ (recovery rate)')
ax.set_ylabel('Density')
ax.set_title('γ Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# R0 distribution
ax = axes[1, 1]
R0_samples = final_samples['beta'] / final_samples['gamma']
R0_true = beta_true / gamma_true
ax.hist(R0_samples, bins=20, alpha=0.7, color='purple', density=True)
ax.axvline(R0_true, color='red', linestyle='--', linewidth=2, label=f'True R₀ = {R0_true:.2f}')
ax.axvline(R0_samples.median(), color='green', linestyle='-', linewidth=2,
           label=f'Estimated R₀ = {R0_samples.median():.2f}')
ax.set_xlabel('R₀ (basic reproduction number)')
ax.set_ylabel('Density')
ax.set_title('R₀ Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Validate Results with Forward Simulation

Let's run the model with our estimated parameters and compare to the observed data.

In [ ]:
# Select a few representative parameter sets from plausible region
n_validation_runs = 20
validation_indices = np.random.choice(len(final_samples), size=n_validation_runs, replace=False)
validation_samples = final_samples.iloc[validation_indices]

# Run forward simulations
validation_results = []
for _, row in validation_samples.iterrows():
    model = SIR(
        beta=row['beta'],
        gamma=row['gamma'],
        s0=population_size - initial_infected,
        i0=initial_infected
    )
    incidence = model.get_incidence()
    validation_results.append(incidence)

# Plot validation results
plt.figure(figsize=(12, 8))

# Plot individual validation runs
days = range(len(incidence_obs))
for i, incidence in enumerate(validation_results):
    alpha = 0.3 if i > 0 else 0.3
    color = 'gray' if i > 0 else 'gray'
    label = 'Plausible simulations' if i == 0 else None
    plt.plot(days, incidence, color=color, alpha=alpha, linewidth=1, label=label)

# Plot observed data
plt.plot(days, incidence_obs, 'ro-', linewidth=3, markersize=6, label='Observed data')

# Calculate and plot ensemble statistics
validation_array = np.array(validation_results)
mean_trajectory = validation_array.mean(axis=0)
std_trajectory = validation_array.std(axis=0)

plt.plot(days, mean_trajectory, 'b-', linewidth=2, label='Plausible mean')
plt.fill_between(days, 
                mean_trajectory - 2*std_trajectory, 
                mean_trajectory + 2*std_trajectory,
                alpha=0.2, color='blue', label='95% prediction interval')

plt.xlabel('Days')
plt.ylabel('Daily Incidence')
plt.title('Model Validation: Plausible Trajectories vs Observed Data')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Calculate validation metrics
rmse_values = []
for incidence in validation_results:
    rmse = np.sqrt(np.mean((incidence - incidence_obs)**2))
    rmse_values.append(rmse)

print(f"\n📊 Validation Metrics:")
print(f"  Mean RMSE: {np.mean(rmse_values):.1f} ± {np.std(rmse_values):.1f}")
print(f"  Best RMSE: {np.min(rmse_values):.1f}")
print(f"  Worst RMSE: {np.max(rmse_values):.1f}")

# Check if observed data falls within prediction intervals
lower_bound = mean_trajectory - 2*std_trajectory
upper_bound = mean_trajectory + 2*std_trajectory
within_bounds = np.sum((incidence_obs >= lower_bound) & (incidence_obs <= upper_bound))
coverage = within_bounds / len(incidence_obs)

print(f"  95% prediction interval coverage: {coverage:.1%}")
if coverage >= 0.9:
    print(f"  ✅ Excellent coverage - model captures observed data well")
elif coverage >= 0.8:
    print(f"  ✅ Good coverage - model mostly captures observed data")
else:
    print(f"  ⚠️  Poor coverage - model may need refinement")

## Interactive History Matching Example

Let's demonstrate the interactive workflow where we can inspect and control each iteration.

In [ ]:
print("🎮 Interactive History Matching Example")
print("=" * 50)

# Create a new engine for interactive demonstration
interactive_engine = hm.quick_setup(
    parameter_bounds=parameter_bounds,
    observations={
        'peak_incidence': (max(incidence_obs), 50**2),
        'total_cases': (sum(incidence_obs), 200**2)
    },
    n_samples=300,
    max_iterations=3,
    sampling_strategy='lhs',
    emulator_type='gpr',
    random_seed=456
)

interactive_engine.set_simulation_function(sir_simulation_function)

print(f"Interactive engine created with {len(interactive_engine._observations.get_feature_names())} features")
print(f"Starting parameter space: β ∈ {parameter_bounds['beta']}, γ ∈ {parameter_bounds['gamma']}")

In [ ]:
# Step 1: Run first iteration
print("\n🔄 Step 1: Running first iteration...")
result1 = interactive_engine.step()

print(f"✅ Iteration 1 completed:")
print(f"  - Generated {len(result1.samples)} plausible samples")
print(f"  - Selected features: {result1.selected_features}")
print(f"  - Parameter ranges:")
print(f"    β: [{result1.samples['beta'].min():.3f}, {result1.samples['beta'].max():.3f}]")
print(f"    γ: [{result1.samples['gamma'].min():.3f}, {result1.samples['gamma'].max():.3f}]")

# At this point, we could inspect the emulator quality, check diagnostics, etc.
# For now, let's accept the iteration
print(f"\n✅ Accepting iteration 1...")
interactive_engine.commit_step()

In [ ]:
# Step 2: Modify strategy and run again
print("\n🔄 Step 2: Modifying strategy for next iteration...")

# Let's focus on just one feature this time
interactive_engine.update_feature_selection(['total_cases'])
print(f"Updated feature selection to focus on total_cases")

result2 = interactive_engine.step()

print(f"✅ Iteration 2 completed:")
print(f"  - Generated {len(result2.samples)} plausible samples")
print(f"  - Acceptance rate: {interactive_engine.acceptance_rate:.3f}")
print(f"  - Selected features: {result2.selected_features}")
print(f"  - Parameter ranges:")
print(f"    β: [{result2.samples['beta'].min():.3f}, {result2.samples['beta'].max():.3f}]")
print(f"    γ: [{result2.samples['gamma'].min():.3f}, {result2.samples['gamma'].max():.3f}]")

print(f"\n✅ Accepting iteration 2...")
interactive_engine.commit_step()

print(f"\n🎯 Interactive workflow completed!")
print(f"Final state: {interactive_engine.current_iteration} iterations completed")
print(f"Total accepted samples: {interactive_engine.progress.total_samples_accepted}")

## Summary

This notebook demonstrated:

1. **Model Setup**: Creating a stochastic SIR model and simulation function
2. **Data Generation**: Creating synthetic "observed" data with known true parameters
3. **Automated Workflow**: Using `hm.quick_setup()` and `engine.run()` for full automation
4. **Result Analysis**: Examining parameter recovery and model validation
5. **Interactive Workflow**: Using `engine.step()` and `engine.commit_step()` for controlled execution

### Key Benefits of the OOP API:

- **Simple Setup**: `quick_setup()` handles common configurations
- **Flexible Control**: Interactive workflow allows inspection and modification
- **Rich Objects**: Results contain comprehensive information about each iteration
- **Modern Patterns**: Clean imports, method chaining, and intuitive interfaces

### Next Steps:

- Try different emulator types (`'linear'`, `'neural_network'`)
- Experiment with feature selection strategies
- Use real observational data instead of synthetic data
- Explore multi-objective calibration with more complex models